<a href="https://colab.research.google.com/github/SamDerden/Fraud_Detection_with-XAI_and_LLM/blob/main/%D0%A1%D0%B5%D0%BC%D0%B8%D0%BD%D0%B0%D1%80_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fastapi uvicorn scikit-learn pydantic nest_asyncio -q

from __future__ import annotations

from datetime import datetime
from typing import Literal, Optional

import numpy as np
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field, field_validator
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

MODEL_VERSION = "2026-09-30"
FEATURE_NAMES = [
    "amount_log", "hour", "day_of_week", "tx_count_24h", "tx_count_7d",
    "distinct_merchants_7d", "distinct_countries_30d", "device_age_days",
    "is_new_device", "is_night_tx", "avg_amount_ratio", "channel_mobile",
]

def _train_model():
    X, y = make_classification(
        n_samples=4000, n_features=12, n_informative=8, n_redundant=2,
        weights=[0.85, 0.15], flip_y=0.02, random_state=42,
    )
    X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    m = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    m.fit(X_train, y_train)
    return m

model = _train_model()

class Geo(BaseModel):
    country: Optional[str] = Field(None, examples=["RU"])
    city: Optional[str] = Field(None, examples=["Kazan"])

class ScoreRequest(BaseModel):
    transaction_id: str = Field(..., examples=["txn_839201"])
    user_id: str = Field(..., examples=["usr_10482"])
    amount: float = Field(..., gt=0, examples=[15700.50])
    currency: str = Field("RUB", examples=["RUB"])
    merchant_id: Optional[str] = Field(None, examples=["mrc_5521"])
    device_id: Optional[str] = Field(None, examples=["dev_9f3a1c"])
    geo: Optional[Geo] = None
    timestamp: str = Field(..., examples=["2026-09-14T03:32:00Z"])
    hour: Optional[int] = Field(None, ge=0, le=23)
    day_of_week: Optional[int] = Field(None, ge=0, le=6)
    tx_count_24h: Optional[int] = Field(0, ge=0)
    tx_count_7d: Optional[int] = Field(0, ge=0)
    distinct_merchants_7d: Optional[int] = Field(1, ge=1)
    distinct_countries_30d: Optional[int] = Field(1, ge=1)
    device_age_days: Optional[int] = Field(100, ge=0)
    channel: Optional[str] = Field("mobile", examples=["mobile", "web", "pos", "api"])

    @field_validator("timestamp")
    @classmethod
    def timestamp_not_empty(cls, v: str) -> str:
        if not v or not v.strip():
            raise ValueError("timestamp не может быть пустым")
        return v

class ScoreResponse(BaseModel):
    risk_score: float = Field(..., ge=0.0, le=1.0)
    decision: Literal["approve", "review", "block"]
    explanation: str
    model_version: str
    transaction_id: str

class HealthResponse(BaseModel):
    status: str
    model_version: str
    timestamp: str

class ModelInfoResponse(BaseModel):
    model_version: str
    model_type: str
    n_features: int
    feature_names: list[str]
    decisions: list[str]

class ServiceInfoResponse(BaseModel):
    service: str
    version: str
    endpoints: list[str]
    docs: str

def _build_features(req: ScoreRequest) -> np.ndarray:
    try:
        ts = datetime.fromisoformat(req.timestamp.replace("Z", "+00:00"))
        hour = req.hour if req.hour is not None else ts.hour
        dow = req.day_of_week if req.day_of_week is not None else ts.weekday()
    except Exception:
        hour = req.hour if req.hour is not None else 12
        dow = req.day_of_week if req.day_of_week is not None else 2
    is_new_device = 1 if (req.device_age_days or 100) < 7 else 0
    is_night = 1 if hour <= 5 else 0
    channel_mobile = 1 if (req.channel or "mobile") == "mobile" else 0
    features = np.array([
        np.log1p(req.amount), hour, dow,
        req.tx_count_24h or 0, req.tx_count_7d or 0,
        req.distinct_merchants_7d or 1, req.distinct_countries_30d or 1,
        req.device_age_days or 100, is_new_device, is_night, 1.0, channel_mobile,
    ], dtype=float)
    return features.reshape(1, -1)

def _make_decision(risk_score: float) -> str:
    if risk_score >= 0.75:
        return "block"
    if risk_score >= 0.45:
        return "review"
    return "approve"

def _make_explanation(req: ScoreRequest, risk_score: float, decision: str) -> str:
    reasons = []
    if req.amount >= 10000:
        reasons.append("крупная сумма")
    hour = req.hour
    if hour is None:
        try:
            hour = datetime.fromisoformat(req.timestamp.replace("Z", "+00:00")).hour
        except Exception:
            hour = 12
    if hour <= 5:
        reasons.append("ночное время")
    if (req.device_age_days or 100) < 7:
        reasons.append("новое устройство")
    if (req.tx_count_24h or 0) >= 8:
        reasons.append("много операций за 24ч")
    if (req.distinct_countries_30d or 1) >= 3:
        reasons.append("несколько стран за 30 дней")
    if not reasons:
        return "Признаки соответствуют обычному поведению клиента" if decision == "approve" else f"Риск {risk_score:.2f} — требуется проверка"
    return " + ".join(reasons)

app = FastAPI(
    title="Антифрод-сервис банка",
    description="ML-сервис детекции подозрительных транзакций",
    version="1.0.0",
)

@app.get("/", response_model=ServiceInfoResponse, tags=["service"])
def root():
    return ServiceInfoResponse(
        service="antifraud-scoring", version="1.0.0",
        endpoints=["/", "/health", "/v1/score", "/v1/model-info", "/docs"], docs="/docs",
    )

@app.get("/health", response_model=HealthResponse, tags=["service"])
def health():
    if model is None:
        raise HTTPException(status_code=503, detail="Модель не загружена")
    return HealthResponse(
        status="ok", model_version=MODEL_VERSION,
        timestamp=datetime.utcnow().isoformat() + "Z",
    )

@app.post("/v1/score", response_model=ScoreResponse, tags=["scoring"])
def score_transaction(req: ScoreRequest):
    if model is None:
        raise HTTPException(status_code=503, detail="Сервис временно недоступен")
    features = _build_features(req)
    proba = float(model.predict_proba(features)[0, 1])
    decision = _make_decision(proba)
    explanation = _make_explanation(req, proba, decision)
    return ScoreResponse(
        risk_score=round(proba, 4), decision=decision, explanation=explanation,
        model_version=MODEL_VERSION, transaction_id=req.transaction_id,
    )

@app.get("/v1/model-info", response_model=ModelInfoResponse, tags=["scoring"])
def model_info():
    return ModelInfoResponse(
        model_version=MODEL_VERSION, model_type="RandomForestClassifier",
        n_features=len(FEATURE_NAMES), feature_names=FEATURE_NAMES,
        decisions=["approve", "review", "block"],
    )

print("Приложение создано, модель обучена.")

!pip install pyngrok -q
from pyngrok import ngrok

ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])  # вставьте свой токен

public_url = ngrok.connect(8000)
print("Swagger UI:", public_url.public_url + "/docs")

Приложение создано, модель обучена.
Swagger UI: https://unshafted-monodomous-meredith.ngrok-free.dev/docs
